# Video Face Authentication Pipeline

This notebook wires the whole pipeline together:
- register a user's face embeddings from reference images
- run Buffalo face detection and alignment on a video
- run liveness detection on sampled faces
- generate CNN face embeddings from the video
- compare against the stored user profile
- make a final authentication decision

The matching thresholds are starting values and should be tuned on real verification data.

In [ ]:
from pathlib import Path
from datetime import datetime
import random
from collections import defaultdict

import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from PIL import Image
from tqdm.auto import tqdm

from insightface.app import FaceAnalysis
from insightface.utils import face_align

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

print("device:", device)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
USER_ID = "user_001"
REGISTRATION_IMAGE_DIR = Path(r"C:\DSP\registration_images\user_001")
AUTH_VIDEO_PATH = Path(r"C:\DSP\auth_video.mp4")
PROFILES_DIR = Path(r"C:\DSP\profiles")
PROFILES_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_PATH = PROFILES_DIR / f"{USER_ID}.pt"

LIVENESS_REPO_DIR = Path(r"C:\DSP\face_liveness_vit")
LIVENESS_CHECKPOINT_PATH = LIVENESS_REPO_DIR / "model.pt"
EMBEDDING_CHECKPOINT_PATH = Path(r"C:\DSP\checkpoints\celeba_embedding_best.pt")

DET_SIZE = (640, 640)
MAX_AUTH_FRAMES = 12
MIN_AUTH_FACES = 6
REGISTRATION_MIN_IMAGES = 3
REGISTRATION_OUT_SIZE = 224
AUTH_OUT_SIZE = 224
RANDOM_SEED = 42

LIVENESS_SPOOF_THRESHOLD = 0.1199
MIN_LIVE_RATIO = 0.60
MATCH_THRESHOLD = 0.60
MIN_MATCH_RATIO = 0.50
USE_MEDIAN_SCORE = True

PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if device.type == "cuda" else ['CPUExecutionProvider']

assert LIVENESS_CHECKPOINT_PATH.exists(), LIVENESS_CHECKPOINT_PATH
assert EMBEDDING_CHECKPOINT_PATH.exists(), EMBEDDING_CHECKPOINT_PATH
print("registration dir:", REGISTRATION_IMAGE_DIR)
print("auth video:", AUTH_VIDEO_PATH)
print("profile path:", PROFILE_PATH)
print("embedding checkpoint:", EMBEDDING_CHECKPOINT_PATH)
print("liveness checkpoint:", LIVENESS_CHECKPOINT_PATH)

In [ ]:
# -----------------------------
# Liveness ViT model
# -----------------------------
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=128):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x


class Encoder(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=int(d_model * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        return self.transformer_encoder(x)


class LivenessViT(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        d_model=128,
        nhead=4,
        num_layers=2,
        num_classes=2,
        mlp_ratio=4.0,
        dropout=0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_chans=3, embed_dim=d_model)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, d_model))
        self.pos_drop = nn.Dropout(dropout)
        self.encoder = Encoder(d_model=d_model, nhead=nhead, num_layers=num_layers, mlp_ratio=mlp_ratio, dropout=dropout)
        self.head = nn.Linear(d_model, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        x = self.patch_embed(x)
        batch_size = x.size(0)
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.pos_drop(x)
        x = self.encoder(x)
        cls_out = x[:, 0]
        return self.head(cls_out)

In [ ]:
# -----------------------------
# CNN embedding model
# -----------------------------
class FaceEmbeddingCNN(nn.Module):
    def __init__(self, embedding_dim, num_classes, use_pretrained=False):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
        backbone = models.resnet18(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(in_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding(features)
        normalized_embedding = nn.functional.normalize(embedding, p=2, dim=1)
        logits = self.classifier(embedding)
        return normalized_embedding, logits

In [ ]:
# -----------------------------
# Load models
# -----------------------------
liveness_model = LivenessViT(
    img_size=224,
    patch_size=16,
    d_model=128,
    nhead=4,
    num_layers=2,
    num_classes=2,
).to(device)

liveness_checkpoint = torch.load(LIVENESS_CHECKPOINT_PATH, map_location=device, weights_only=False)
liveness_state = liveness_checkpoint["model"] if isinstance(liveness_checkpoint, dict) and "model" in liveness_checkpoint else liveness_checkpoint
liveness_model.load_state_dict(liveness_state, strict=True)
liveness_model.eval()

embedding_checkpoint = torch.load(EMBEDDING_CHECKPOINT_PATH, map_location=device, weights_only=False)
embedding_dim = embedding_checkpoint.get("embedding_dim", 256)
embedding_image_size = embedding_checkpoint.get("image_size", 224)
num_classes = embedding_checkpoint.get("num_classes", 10177)

embedding_model = FaceEmbeddingCNN(
    embedding_dim=embedding_dim,
    num_classes=num_classes,
    use_pretrained=False,
).to(device)
embedding_model.load_state_dict(embedding_checkpoint["model_state_dict"])
embedding_model.eval()

face_app = FaceAnalysis(name="buffalo_l", providers=PROVIDERS)
face_app.prepare(ctx_id=0 if device.type == "cuda" else -1, det_size=DET_SIZE)

liveness_transform = transforms.Compose([
    transforms.Resize(int(224 * 1.14)),
    transforms.CenterCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

embedding_transform = transforms.Compose([
    transforms.Resize((embedding_image_size, embedding_image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("liveness model loaded")
print("embedding model loaded")
print("embedding dim:", embedding_dim)
print("embedding image size:", embedding_image_size)
print("Buffalo face detector loaded")

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def get_aligned_face(frame_bgr, face_obj, out_size=224):
    if hasattr(face_obj, "kps") and face_obj.kps is not None:
        aligned_bgr = face_align.norm_crop(frame_bgr, landmark=face_obj.kps, image_size=out_size)
        aligned_rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB)
        return Image.fromarray(aligned_rgb)

    x1, y1, x2, y2 = face_obj.bbox.astype(int)
    h, w = frame_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    crop_bgr = frame_bgr[y1:y2, x1:x2]
    if crop_bgr.size == 0:
        return None
    crop_bgr = cv2.resize(crop_bgr, (out_size, out_size))
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(crop_rgb)


def pick_largest_face(faces):
    if not faces:
        return None
    return sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]), reverse=True)[0]


def image_paths_from_dir(folder):
    assert folder.exists(), folder
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])


@torch.no_grad()
def embed_pil_faces(face_images):
    tensors = [embedding_transform(face).unsqueeze(0) for face in face_images]
    batch = torch.cat(tensors, dim=0).to(device)
    embeddings, _ = embedding_model(batch)
    return embeddings.cpu()


@torch.no_grad()
def liveness_on_faces(face_images):
    spoof_probs = []
    live_flags = []
    progress = tqdm(face_images, desc="liveness", dynamic_ncols=True)
    for face in progress:
        x = liveness_transform(face).unsqueeze(0).to(device)
        logits = liveness_model(x)
        probs = torch.softmax(logits, dim=1)
        spoof_prob = probs[0, 1].item()
        is_live = spoof_prob < LIVENESS_SPOOF_THRESHOLD
        spoof_probs.append(spoof_prob)
        live_flags.append(is_live)
        progress.set_postfix({
            "spoof": f"{spoof_prob:.4f}",
            "live": is_live,
        })

    mean_spoof = sum(spoof_probs) / len(spoof_probs)
    live_ratio = sum(live_flags) / len(live_flags)
    session_live = (mean_spoof < LIVENESS_SPOOF_THRESHOLD) and (live_ratio >= MIN_LIVE_RATIO)
    return {
        "spoof_probs": spoof_probs,
        "live_flags": live_flags,
        "mean_spoof": mean_spoof,
        "live_ratio": live_ratio,
        "session_live": session_live,
    }


def collect_registration_faces(image_dir):
    image_paths = image_paths_from_dir(image_dir)
    if len(image_paths) < REGISTRATION_MIN_IMAGES:
        raise RuntimeError(f"Need at least {REGISTRATION_MIN_IMAGES} registration images, found {len(image_paths)}")

    faces = []
    used_paths = []
    progress = tqdm(image_paths, desc="registration faces", dynamic_ncols=True)
    for image_path in progress:
        frame_bgr = cv2.imread(str(image_path))
        if frame_bgr is None:
            continue
        detected = face_app.get(frame_bgr)
        face = pick_largest_face(detected)
        if face is None:
            continue
        aligned = get_aligned_face(frame_bgr, face, out_size=REGISTRATION_OUT_SIZE)
        if aligned is None:
            continue
        faces.append(aligned)
        used_paths.append(str(image_path))
        progress.set_postfix({"kept": len(faces)})

    if len(faces) < REGISTRATION_MIN_IMAGES:
        raise RuntimeError(f"Only {len(faces)} usable registration faces found")
    return used_paths, faces


def random_frame_indices(video_path, max_frames, seed=42):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if total_frames <= 0:
        raise RuntimeError("Could not determine total frame count for random sampling")

    sample_count = min(max_frames, total_frames)
    rng = random.Random(seed)
    return sorted(rng.sample(range(total_frames), sample_count))


def sample_video_faces(video_path, max_frames, seed=42):
    if not Path(video_path).exists():
        raise FileNotFoundError(video_path)

    frame_indices = random_frame_indices(video_path, max_frames=max_frames, seed=seed)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    records = []
    progress = tqdm(frame_indices, desc="video sampling", dynamic_ncols=True)
    for frame_idx in progress:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame_bgr = cap.read()
        if not ok:
            continue

        detected = face_app.get(frame_bgr)
        face = pick_largest_face(detected)
        if face is None:
            continue

        aligned = get_aligned_face(frame_bgr, face, out_size=AUTH_OUT_SIZE)
        if aligned is None:
            continue

        records.append({
            "frame_index": int(frame_idx),
            "face": aligned,
        })
        progress.set_postfix({"faces": len(records)})

    cap.release()
    return records


def build_profile(user_id, image_dir, profile_path):
    used_paths, faces = collect_registration_faces(image_dir)
    embeddings = embed_pil_faces(faces)
    mean_embedding = embeddings.mean(dim=0)
    mean_embedding = mean_embedding / mean_embedding.norm(p=2)

    profile = {
        "user_id": user_id,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "source_paths": used_paths,
        "embeddings": embeddings,
        "mean_embedding": mean_embedding,
        "embedding_dim": embeddings.shape[1],
        "checkpoint_path": str(EMBEDDING_CHECKPOINT_PATH),
    }
    torch.save(profile, profile_path)
    return profile


def load_profile(profile_path):
    profile = torch.load(profile_path, map_location="cpu", weights_only=False)
    profile["embeddings"] = profile["embeddings"].float()
    profile["mean_embedding"] = profile["mean_embedding"].float()
    return profile


def authenticate_video(video_path, profile):
    sampled_records = sample_video_faces(video_path, max_frames=MAX_AUTH_FRAMES, seed=RANDOM_SEED)
    if len(sampled_records) < MIN_AUTH_FACES:
        raise RuntimeError(f"Only {len(sampled_records)} usable faces sampled from video; need at least {MIN_AUTH_FACES}")

    face_images = [record["face"] for record in sampled_records]
    liveness_result = liveness_on_faces(face_images)
    video_embeddings = embed_pil_faces(face_images)

    gallery_embeddings = profile["embeddings"]
    mean_embedding = profile["mean_embedding"].unsqueeze(0)

    similarity_to_gallery = video_embeddings @ gallery_embeddings.T
    similarity_to_mean = (video_embeddings @ mean_embedding.T).squeeze(1)
    per_frame_best = torch.maximum(similarity_to_gallery.max(dim=1).values, similarity_to_mean)

    if USE_MEDIAN_SCORE:
        session_score = per_frame_best.median().item()
    else:
        session_score = per_frame_best.mean().item()

    match_ratio = (per_frame_best >= MATCH_THRESHOLD).float().mean().item()
    authenticated = (
        liveness_result["session_live"]
        and session_score >= MATCH_THRESHOLD
        and match_ratio >= MIN_MATCH_RATIO
    )

    frame_results = []
    for idx, record in enumerate(sampled_records):
        frame_results.append({
            "frame_index": record["frame_index"],
            "spoof_prob": liveness_result["spoof_probs"][idx],
            "is_live": liveness_result["live_flags"][idx],
            "similarity": float(per_frame_best[idx]),
            "matched": bool(per_frame_best[idx] >= MATCH_THRESHOLD),
        })

    return {
        "sampled_faces": len(sampled_records),
        "liveness": liveness_result,
        "session_score": session_score,
        "match_ratio": match_ratio,
        "authenticated": authenticated,
        "frame_results": frame_results,
    }

In [ ]:
# -----------------------------
# Registration flow
# -----------------------------
profile = build_profile(USER_ID, REGISTRATION_IMAGE_DIR, PROFILE_PATH)
print("profile saved:", PROFILE_PATH)
print("user id:", profile["user_id"])
print("registration images used:", len(profile["source_paths"]))
print("embedding shape:", tuple(profile["embeddings"].shape))

In [ ]:
# -----------------------------
# Authentication flow
# -----------------------------
profile = load_profile(PROFILE_PATH)
result = authenticate_video(AUTH_VIDEO_PATH, profile)

print("\n===== AUTHENTICATION SUMMARY =====")
print("sampled faces        :", result["sampled_faces"])
print("session live         :", result["liveness"]["session_live"])
print("mean spoof prob      :", f"{result['liveness']['mean_spoof']:.4f}")
print("live ratio           :", f"{result['liveness']['live_ratio']:.2%}")
print("session similarity   :", f"{result['session_score']:.4f}")
print("match ratio          :", f"{result['match_ratio']:.2%}")
print("authenticated        :", result["authenticated"])

for row in result["frame_results"]:
    print(
        f"frame={row['frame_index']:05d} | "
        f"spoof={row['spoof_prob']:.4f} | "
        f"live={row['is_live']} | "
        f"sim={row['similarity']:.4f} | "
        f"matched={row['matched']}"
    )